# AD Criteria Verification and Extraction

This notebook identifies AD visits in the longitudinal DELCODE metadata using a standardized criteria format.

## Criteria Table

| Cohort | Cognitive Rule | Biomarker Rule | Decision |
|---|---|---|---|
| AD (strict) | `mmstot <= 24` AND `cdrglobal >= 1` | `ratio_Abeta42_40 <= 0.08` AND `ratio_Abeta42_phosphotau181 < 9.68` | Included in strict AD output |
| AD (cognitive-only) | `mmstot <= 24` AND `cdrglobal >= 1` | Not required | Included in cognitive-only AD output |
| Exclusion rule | `mmstot > 24` AND `cdrglobal < 1` | OR `ratio_Abeta42_40 > 0.08` AND `ratio_Abeta42_phosphotau181 >= 9.68` | Not classified as AD |

## Cognitive Decline Validation (Post-filter)

For patients who were previously classified as MCI (i.e., converters), we additionally require that **cognitive scores must strictly worsen** at the AD visit compared to the **visit immediately before the first AD visit** (regardless of that visit's classification):

- `mmstot` must **strictly decrease** (lower = worse)
- `cdrglobal` must **strictly increase** (higher = worse)

**Note:** We only check worsening on **cognitive scores** (`mmstot`, `cdrglobal`). Biomarker values are not checked for worsening.

AD visits that fail this check are excluded. Patients without prior MCI visits are not affected by this rule.

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Centralized paths ──────────────────────────────────────────────────
BASE = Path('/dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION')
INTERMEDIATE = BASE / 'data' / 'intermediate'

input_csv = INTERMEDIATE / 'combined' / 'all_visits.csv'
output_dir = INTERMEDIATE / 'ad'
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / 'strict_all_visits.csv'
output_file_cognitive = output_dir / 'cognitive_all_visits.csv'
output_file_ad_first = output_dir / 'strict_first_visit.csv'
output_file_cognitive_first = output_dir / 'cognitive_first_visit.csv'
output_excel = output_dir / 'highlighted.xlsx'

# Load data
df = pd.read_csv(input_csv)

print(f"Input CSV: {input_csv}")
print(f"Output directory: {output_dir}")
print(f"Total rows: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head(3)


Input CSV: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/combined/all_visits.csv
Output directory: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/ad
Total rows: 4605

Columns: ['file', 'Repseudonym', 'visdate', 'sex', 'age', 'prmdiag', 'mmstot', 'cdrtot', 'cdrglobal', 'Abeta38', 'Abeta40', 'Abeta42', 'totaltau', 'phosphotau181', 'ratio_Abeta42_40', 'ratio_Abeta42_phosphotau181', 'visnam', 'scan_date', 'scan_year']

First few rows:


,file,Repseudonym,visdate,sex,age,prmdiag,mmstot,cdrtot,cdrglobal,Abeta38,Abeta40,Abeta42,totaltau,phosphotau181,ratio_Abeta42_40,ratio_Abeta42_phosphotau181,visnam,scan_date,scan_year
0,BASELINE,011d501d1,01-10-2015,f,70.08,0,30,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,08-10-2015,M0
1,FOLLOWUP,011d501d1,12-10-2016,f,71.11,0,30,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Follow-up 1,11-11-2016,M12
2,FOLLOWUP,011d501d1,20-09-2017,f,72.05,0,30,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Follow-up 2,NaN,NaN


## Missing Values and Format Check

In [6]:
# Check for missing values in key columns
key_cols = ['mmstot', 'cdrglobal', 'ratio_Abeta42_40', 'ratio_Abeta42_phosphotau181']
print("Missing values in key columns:")
for col in key_cols:
    missing = df[col].isna().sum()
    print(f"  {col}: {missing} ({100*missing/len(df):.1f}%)")

# Convert key columns to numeric
df['mmstot'] = pd.to_numeric(df['mmstot'], errors='coerce')
df['cdrglobal'] = pd.to_numeric(df['cdrglobal'], errors='coerce')
df['ratio_Abeta42_40'] = pd.to_numeric(df['ratio_Abeta42_40'], errors='coerce')
df['ratio_Abeta42_phosphotau181'] = pd.to_numeric(df['ratio_Abeta42_phosphotau181'], errors='coerce')

# Has all required data (no nulls in key columns)
has_data = (df['mmstot'].notna() & 
            df['cdrglobal'].notna() & 
            df['ratio_Abeta42_40'].notna() & 
            df['ratio_Abeta42_phosphotau181'].notna())

print(f"Rows with all required data: {has_data.sum()}")

Missing values in key columns:
  mmstot: 99 (2.1%)
  cdrglobal: 67 (1.5%)
  ratio_Abeta42_40: 3829 (83.1%)
  ratio_Abeta42_phosphotau181: 3830 (83.2%)
Rows with all required data: 760


## Extract All Visits

In [7]:
# ── Strict AD: cognitive + biomarker criteria ──
meets_cognitive_and_biomarkers_criteria = (
    (df['mmstot'] <= 24) & 
    (df['cdrglobal'] >= 1) & 
    (df['ratio_Abeta42_40'] <= 0.08) & 
    (df['ratio_Abeta42_phosphotau181'] < 9.68)
)

# ── Cognitive-only AD criteria ──
meets_cognitive_criteria = (
    (df['mmstot'] <= 24) & 
    (df['cdrglobal'] >= 1)
)

# Exclusion Criteria
falls_out_criteria = (
    ((df['ratio_Abeta42_40'] > 0.08) & (df['ratio_Abeta42_phosphotau181'] >= 9.68)) |
    ((df['mmstot'] > 24) & (df['cdrglobal'] < 1))
)

# ── Store indices for reuse ──
idx_strict_all = set(df[meets_cognitive_and_biomarkers_criteria].index)
idx_cognitive_only_all = set(df[meets_cognitive_criteria & ~meets_cognitive_and_biomarkers_criteria].index)

print(f"Rows meeting STRICT AD criteria (cognitive + biomarkers): {len(idx_strict_all)}")
print(f"Rows meeting COGNITIVE-ONLY AD criteria: {len(idx_strict_all) + len(idx_cognitive_only_all)}")
print(f"  → Cognitive-only (not strict): {len(idx_cognitive_only_all)}")
print(f"Rows meeting exclusion criteria: {falls_out_criteria.sum()}")

# Extract dataframes
df_ad = df[meets_cognitive_and_biomarkers_criteria].copy()
df_ad_cognitive = df[meets_cognitive_criteria].copy()

print(f"\n=== EXTRACTION RESULT ===")
print(f"Strict AD rows: {len(df_ad)}")
print(f"Cognitive-only AD rows: {len(df_ad_cognitive)}")
df_ad.head(3)


Rows meeting STRICT AD criteria (cognitive + biomarkers): 36
Rows meeting COGNITIVE-ONLY AD criteria: 274
  → Cognitive-only (not strict): 238
Rows meeting exclusion criteria: 3885

=== EXTRACTION RESULT ===
Strict AD rows: 36
Cognitive-only AD rows: 274


,file,Repseudonym,visdate,sex,age,prmdiag,mmstot,cdrtot,cdrglobal,Abeta38,Abeta40,Abeta42,totaltau,phosphotau181,ratio_Abeta42_40,ratio_Abeta42_phosphotau181,visnam,scan_date,scan_year
94,FOLLOWUP,0588be6c5,04-05-2021,f,69.01,2,22.0,5.5,1.0,2507.708783,7909.621307,455.307315,689.000000,92.205,0.057564,4.937989,Follow-up 3,NaN,NaN
184,FOLLOWUP,0df733308,28-11-2017,f,78.41,2,18.0,8,1.0,2321.484400,7875.338698,462.880155,456.722178,79.917,0.058776,5.792011,Follow-up 3,28-11-2017,M36
449,BASELINE,1b6c308bb,30-09-2014,f,78.58,5,19.0,7,1.0,3197.564219,7543.335582,343.567648,994.110375,118.000,0.045546,2.911590,NaN,08-10-2014,M0


## Post-filter: Cognitive Decline Validation

For patients with prior MCI visits, exclude AD visits where cognitive scores did NOT strictly worsen compared to the **visit immediately before the first AD visit** (in the full dataset, regardless of MCI classification).

- `mmstot` must decrease (lower = worse)
- `cdrglobal` must increase (higher = worse)

**Note:** Only cognitive scores are checked for worsening. Patients without prior MCI visits are unaffected.

In [8]:
# ── Post-filter: Cognitive Decline Validation ──────────────────────────
# For MCI converters, AD visits must show strictly worse cognitive scores
# compared to the visit immediately before the first AD visit (in the full
# dataset, regardless of MCI classification).
# Rule: mmstot must DECREASE and cdrglobal must INCREASE.

# Load MCI visits (to identify which patients are converters)
df_mci = pd.read_csv(INTERMEDIATE / 'mci' / 'all_visits.csv')
mci_patients = set(df_mci['Repseudonym'].unique())
print(f'MCI patients loaded: {len(mci_patients)}')

# Use the full dataset (df) to find the visit immediately before AD
df['_vd'] = pd.to_datetime(df['visdate'], dayfirst=True, errors='coerce')
df_ad['_vd'] = pd.to_datetime(df_ad['visdate'], dayfirst=True, errors='coerce')
df_ad_cognitive['_vd'] = pd.to_datetime(df_ad_cognitive['visdate'], dayfirst=True, errors='coerce')

def get_visit_before_first_ad(patient_id, ad_df, full_df):
    """Get the visit immediately before the patient's first AD visit from the full dataset."""
    patient_ad = ad_df[ad_df['Repseudonym'] == patient_id].sort_values('_vd')
    if patient_ad.empty:
        return None
    first_ad_date = patient_ad['_vd'].iloc[0]
    
    # Find all visits for this patient BEFORE the first AD visit (from full dataset)
    patient_all = full_df[full_df['Repseudonym'] == patient_id].sort_values('_vd')
    visits_before_ad = patient_all[patient_all['_vd'] < first_ad_date]
    if visits_before_ad.empty:
        return None
    return visits_before_ad.iloc[-1]  # the visit immediately before first AD

def apply_decline_filter(ad_df, full_df, label):
    """Remove AD visits that don't show strict cognitive decline vs visit before first AD."""
    ad_patients_in_mci = set(ad_df['Repseudonym'].unique()) & mci_patients
    print(f'\n--- {label} ---')
    print(f'AD patients also in MCI: {len(ad_patients_in_mci)}')
    
    exclude_indices = []
    checked = 0
    for pid in sorted(ad_patients_in_mci):
        ref_visit = get_visit_before_first_ad(pid, ad_df, full_df)
        if ref_visit is None:
            continue  # no visit before AD, skip
        checked += 1
        ref_mmstot = ref_visit['mmstot']
        ref_cdrglobal = ref_visit['cdrglobal']
        
        patient_ad = ad_df[ad_df['Repseudonym'] == pid]
        for idx, row in patient_ad.iterrows():
            mmstot_worse = row['mmstot'] < ref_mmstot   # must decrease
            cdrglobal_worse = row['cdrglobal'] > ref_cdrglobal  # must increase
            if not (mmstot_worse and cdrglobal_worse):
                exclude_indices.append(idx)
                print(f'  EXCLUDE {pid}: AD mmstot={row["mmstot"]}, cdrglobal={row["cdrglobal"]}'
                      f' vs ref mmstot={ref_mmstot}, cdrglobal={ref_cdrglobal} (ref date: {ref_visit["visdate"]})')
    
    print(f'Patients checked: {checked}')
    print(f'AD visits excluded: {len(exclude_indices)}')
    
    filtered = ad_df.drop(index=exclude_indices)
    print(f'Remaining AD visits: {len(ad_df)} → {len(filtered)}')
    return filtered, set(exclude_indices)

# Apply filter to both AD datasets
df_ad, excluded_strict = apply_decline_filter(df_ad, df, 'Strict AD')
df_ad_cognitive, excluded_cognitive = apply_decline_filter(df_ad_cognitive, df, 'Cognitive-only AD')

# Clean up temp columns
df = df.drop(columns=['_vd'])
df_ad = df_ad.drop(columns=['_vd'])
df_ad_cognitive = df_ad_cognitive.drop(columns=['_vd'])

# Update index sets
idx_strict_all -= excluded_strict
idx_cognitive_only_all -= excluded_cognitive

print(f'\n=== AFTER COGNITIVE DECLINE FILTER ===')
print(f'Strict AD rows: {len(df_ad)}')
print(f'Cognitive-only AD rows: {len(df_ad_cognitive)}')

MCI patients loaded: 490

--- Strict AD ---
AD patients also in MCI: 7
  EXCLUDE 0df733308: AD mmstot=18.0, cdrglobal=1.0 vs ref mmstot=20.0, cdrglobal=1.0 (ref date: 15-11-2016)
  EXCLUDE b18d729af: AD mmstot=23.0, cdrglobal=1.0 vs ref mmstot=29.0, cdrglobal=nan (ref date: 19-06-2017)
  EXCLUDE d0887896f: AD mmstot=19.0, cdrglobal=1.0 vs ref mmstot=nan, cdrglobal=1.0 (ref date: 29-03-2021)
Patients checked: 7
AD visits excluded: 3
Remaining AD visits: 36 → 33

--- Cognitive-only AD ---
AD patients also in MCI: 59
  EXCLUDE 0df733308: AD mmstot=20.0, cdrglobal=1.0 vs ref mmstot=20.0, cdrglobal=0.5 (ref date: 23-11-2015)
  EXCLUDE 1169cb29f: AD mmstot=23.0, cdrglobal=1.0 vs ref mmstot=25.0, cdrglobal=1.0 (ref date: 09-12-2021)
  EXCLUDE 1eebab647: AD mmstot=21.0, cdrglobal=1.0 vs ref mmstot=26.0, cdrglobal=1.0 (ref date: 21-08-2017)
  EXCLUDE 1eebab647: AD mmstot=17.0, cdrglobal=1.0 vs ref mmstot=26.0, cdrglobal=1.0 (ref date: 21-08-2017)
  EXCLUDE 34f06b217: AD mmstot=17.0, cdrglobal=1

In [9]:
# Save extracted rows to CSV
df_ad.to_csv(output_file, index=False)
print(f"✓ Extracted AD rows saved to: {output_file}")
print(f"  Total rows: {len(df_ad)}")
print(f"  File size: {output_file.stat().st_size:,} bytes")

✓ Extracted AD rows saved to: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/ad/strict_all_visits.csv
  Total rows: 33
  File size: 5,923 bytes


## Check For Completeness

In [10]:
# Summary and verification
print("=" * 70)
print("AD CRITERIA VERIFICATION SUMMARY")
print("=" * 70)
print(f"\nTotal rows in dataset: {len(df):,}")
print(f"Rows with all required biomarkers: {has_data.sum():,}")
print(f"  → Missing ratio_Abeta42_40 or ratio_Abeta42_phosphotau181: {(~has_data).sum():,}")

print(f"\n>>> STRICT AD (cognitive + biomarkers): {len(idx_strict_all)}")
if len(df_ad) > 0:
    print(f"  ✓ mmstot (MMSE) <= 24: All {(df_ad['mmstot'] <= 24).all()}")
    print(f"  ✓ cdrglobal (CDR) >= 1: All {(df_ad['cdrglobal'] >= 1).all()}")
    print(f"  ✓ ratio_Abeta42_40 <= 0.08: All {(df_ad['ratio_Abeta42_40'] <= 0.08).all()}")
    print(f"  ✓ ratio_Abeta42_phosphotau181 < 9.68: All {(df_ad['ratio_Abeta42_phosphotau181'] < 9.68).all()}")
    
    print(f"\nStrict AD rows statistics:")
    print(f"  mmstot range: {df_ad['mmstot'].min():.0f} - {df_ad['mmstot'].max():.0f}")
    print(f"  cdrglobal range: {df_ad['cdrglobal'].min():.2f} - {df_ad['cdrglobal'].max():.2f}")
    print(f"  ratio_Abeta42_40 range: {df_ad['ratio_Abeta42_40'].min():.4f} - {df_ad['ratio_Abeta42_40'].max():.4f}")
    print(f"  ratio_Abeta42_phosphotau181 range: {df_ad['ratio_Abeta42_phosphotau181'].min():.2f} - {df_ad['ratio_Abeta42_phosphotau181'].max():.2f}")

print(f"\n>>> COGNITIVE-ONLY AD: {len(idx_cognitive_only_all)} additional rows")
print(f"Rows meeting EXCLUSION criteria: {falls_out_criteria.sum():,}")
print(f"\nOutput file: {output_file}")


AD CRITERIA VERIFICATION SUMMARY

Total rows in dataset: 4,605
Rows with all required biomarkers: 760
  → Missing ratio_Abeta42_40 or ratio_Abeta42_phosphotau181: 3,845

>>> STRICT AD (cognitive + biomarkers): 33
  ✓ mmstot (MMSE) <= 24: All True
  ✓ cdrglobal (CDR) >= 1: All True
  ✓ ratio_Abeta42_40 <= 0.08: All True
  ✓ ratio_Abeta42_phosphotau181 < 9.68: All True

Strict AD rows statistics:
  mmstot range: 14 - 24
  cdrglobal range: 1.00 - 2.00
  ratio_Abeta42_40 range: 0.0287 - 0.0671
  ratio_Abeta42_phosphotau181 range: 1.48 - 8.07

>>> COGNITIVE-ONLY AD: 202 additional rows
Rows meeting EXCLUSION criteria: 3,885

Output file: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/ad/strict_all_visits.csv


## Alternative Analysis: Cognitive Scores Only

Ignore missing biomarkers.

### AD Criteria (Cognitive-Only):
- **(MMSE or mmstot) <= 24** AND
- **(CDR or cdrglobal) >= 1**

Missing biomarker ratios are **ignored** in this analysis.

In [11]:
# Cognitive-Only AD Analysis
# Uses meets_cognitive_criteria and df_ad_cognitive computed above

print(f"Rows meeting COGNITIVE-ONLY AD criteria: {meets_cognitive_criteria.sum()}")

print(f"\n=== COGNITIVE-ONLY AD ===")
print(f"Total rows: {len(df_ad_cognitive)}")
print(f"  → Also meet strict criteria (biomarkers): {len(idx_strict_all)}")
print(f"  → Cognitive-only (missing/non-qualifying biomarkers): {len(idx_cognitive_only_all)}")

print(f"\nCognitive-Only AD rows statistics:")
print(f"  mmstot range: {df_ad_cognitive['mmstot'].min():.0f} - {df_ad_cognitive['mmstot'].max():.0f}")
print(f"  cdrglobal range: {df_ad_cognitive['cdrglobal'].min():.2f} - {df_ad_cognitive['cdrglobal'].max():.2f}")

Rows meeting COGNITIVE-ONLY AD criteria: 274

=== COGNITIVE-ONLY AD ===
Total rows: 236
  → Also meet strict criteria (biomarkers): 33
  → Cognitive-only (missing/non-qualifying biomarkers): 202

Cognitive-Only AD rows statistics:
  mmstot range: 2 - 24
  cdrglobal range: 1.00 - 3.00


In [12]:
# Save cognitive-only AD rows to CSV
df_ad_cognitive.to_csv(output_file_cognitive, index=False)
print(f"✓ Cognitive-only AD rows saved to: {output_file_cognitive}")
print(f"  Total rows: {len(df_ad_cognitive)}")
print(f"  File size: {output_file_cognitive.stat().st_size:,} bytes")

✓ Cognitive-only AD rows saved to: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/ad/cognitive_all_visits.csv
  Total rows: 236
  File size: 22,282 bytes


In [13]:
# Comparison: Biomarker-based vs Cognitive-only
print("\n" + "=" * 70)
print("COMPARISON: BIOMARKER-BASED VS COGNITIVE-ONLY")
print("=" * 70)

print(f"\n1. STRICT AD (cognitive + biomarker criteria):")
print(f"   Rows: {len(df_ad)}")
print(f"   Criteria: mmstot ≤ 24 AND cdrglobal ≥ 1 AND ratio_Abeta42_40 ≤ 0.08 AND ratio_Abeta42_phosphotau181 < 9.68")

print(f"\n2. COGNITIVE-ONLY AD:")
print(f"   Rows: {len(df_ad_cognitive)}")
print(f"   Criteria: mmstot ≤ 24 AND cdrglobal ≥ 1")

print(f"\n3. DIFFERENCE:")
print(f"   Additional rows with cognitive scores but missing/invalid biomarkers: {len(df_ad_cognitive) - len(df_ad)}")

print(f"\nOutput files:")
print(f"  - Strict: {output_file}")
print(f"  - Cognitive-only: {output_file_cognitive}")



COMPARISON: BIOMARKER-BASED VS COGNITIVE-ONLY

1. STRICT AD (cognitive + biomarker criteria):
   Rows: 33
   Criteria: mmstot ≤ 24 AND cdrglobal ≥ 1 AND ratio_Abeta42_40 ≤ 0.08 AND ratio_Abeta42_phosphotau181 < 9.68

2. COGNITIVE-ONLY AD:
   Rows: 236
   Criteria: mmstot ≤ 24 AND cdrglobal ≥ 1

3. DIFFERENCE:
   Additional rows with cognitive scores but missing/invalid biomarkers: 203

Output files:
  - Strict: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/ad/strict_all_visits.csv
  - Cognitive-only: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/ad/cognitive_all_visits.csv


## First Visit Only Analysis

Extract only the first AD visit for each patient from both datasets.

Filtering logic: For each unique patient (Repseudonym), keep only the earliest visit (by visdateeeee).

In [14]:
# First Visit Only - Strict AD
# Parse visdate properly before sorting (DD-MM-YYYY → datetime)
df_ad['_vd'] = pd.to_datetime(df_ad['visdate'], dayfirst=True, errors='coerce')

df_ad_first_visit = (df_ad
                     .sort_values(['Repseudonym', '_vd'])
                     .drop_duplicates(subset=['Repseudonym'], keep='first')
                     .copy())
df_ad = df_ad.drop(columns=['_vd'])
df_ad_first_visit = df_ad_first_visit.drop(columns=['_vd'])

# Store first-visit indices
idx_strict_first = set(df_ad_first_visit.index)

print(f"Strict AD - First Visit Only:")
print(f"  Total patients: {len(df_ad_first_visit)}")
print(f"  Original total rows (all visits): {len(df_ad)}")
print(f"  Patients with multiple visits: {len(df_ad) - len(df_ad_first_visit)}")


Strict AD - First Visit Only:
  Total patients: 28
  Original total rows (all visits): 33
  Patients with multiple visits: 5


In [15]:
# First Visit Only - Cognitive-Only AD
# Parse visdate properly before sorting (DD-MM-YYYY → datetime)
df_ad_cognitive['_vd'] = pd.to_datetime(df_ad_cognitive['visdate'], dayfirst=True, errors='coerce')

df_ad_cognitive_first_visit = (df_ad_cognitive
                               .sort_values(['Repseudonym', '_vd'])
                               .drop_duplicates(subset=['Repseudonym'], keep='first')
                               .copy())
df_ad_cognitive = df_ad_cognitive.drop(columns=['_vd'])
df_ad_cognitive_first_visit = df_ad_cognitive_first_visit.drop(columns=['_vd'])

# Store first-visit indices (cognitive-only = not already in strict)
idx_cognitive_only_first = set(df_ad_cognitive_first_visit.index) - idx_strict_first

print(f"\nCognitive-Only AD - First Visit Only:")
print(f"  Total patients: {len(df_ad_cognitive_first_visit)}")
print(f"  Original total rows (all visits): {len(df_ad_cognitive)}")
print(f"  Patients with multiple visits: {len(df_ad_cognitive) - len(df_ad_cognitive_first_visit)}")



Cognitive-Only AD - First Visit Only:
  Total patients: 108
  Original total rows (all visits): 236
  Patients with multiple visits: 128


## Add first_ad_visit Column
Set `first_ad_visit` to the `scan_date` of the earliest AD visit per patient.

In [16]:
# ── Add first_ad_visit column (scan_date with visdate fallback) ────────
# For each patient, first_ad_visit = scan_date of earliest AD visit,
# falling back to visdate if scan_date is missing. Format: DD-MM-YYYY.

def get_first_visit_date(df):
    """Get earliest visit date per patient, preferring scan_date, falling back to visdate."""
    tmp = df.copy()
    tmp['_date'] = pd.to_datetime(tmp['scan_date'], dayfirst=True, errors='coerce')
    tmp['_date'] = tmp['_date'].fillna(pd.to_datetime(tmp['visdate'], dayfirst=True, errors='coerce'))
    return (tmp.dropna(subset=['_date'])
            .sort_values('_date')
            .drop_duplicates(subset=['Repseudonym'], keep='first')
            .set_index('Repseudonym')['_date'])

first_ad_strict = get_first_visit_date(df_ad)
first_ad_cog = get_first_visit_date(df_ad_cognitive)

# Map back — all visits
df_ad['first_ad_visit'] = df_ad['Repseudonym'].map(first_ad_strict).dt.strftime('%d-%m-%Y')
df_ad_cognitive['first_ad_visit'] = df_ad_cognitive['Repseudonym'].map(first_ad_cog).dt.strftime('%d-%m-%Y')

# Map back — first visit DataFrames
df_ad_first_visit['first_ad_visit'] = df_ad_first_visit['Repseudonym'].map(first_ad_strict).dt.strftime('%d-%m-%Y')
df_ad_cognitive_first_visit['first_ad_visit'] = df_ad_cognitive_first_visit['Repseudonym'].map(first_ad_cog).dt.strftime('%d-%m-%Y')

print(f'first_ad_visit (strict) populated: {df_ad["first_ad_visit"].notna().sum()} / {len(df_ad)} rows')
print(f'first_ad_visit (cognitive) populated: {df_ad_cognitive["first_ad_visit"].notna().sum()} / {len(df_ad_cognitive)} rows')
print(f'Sample (strict):')
df_ad[['Repseudonym', 'visdate', 'scan_date', 'first_ad_visit']].head(5)

# Re-save all-visits CSVs with first_ad_visit column
df_ad.to_csv(output_file, index=False)
df_ad_cognitive.to_csv(output_file_cognitive, index=False)
print(f'\n✓ AD all-visits CSVs re-saved with first_ad_visit column')


first_ad_visit (strict) populated: 33 / 33 rows
first_ad_visit (cognitive) populated: 236 / 236 rows
Sample (strict):

✓ AD all-visits CSVs re-saved with first_ad_visit column


In [17]:
# Save First Visit Only files
df_ad_first_visit.to_csv(output_file_ad_first, index=False)
df_ad_cognitive_first_visit.to_csv(output_file_cognitive_first, index=False)

print(f"\n✓ First visit files saved:")
print(f"  Biomarker-based: {output_file_ad_first}")
print(f"    Total rows: {len(df_ad_first_visit)}")
print(f"    File size: {output_file_ad_first.stat().st_size:,} bytes")
print(f"\n  Cognitive-only: {output_file_cognitive_first}")
print(f"    Total rows: {len(df_ad_cognitive_first_visit)}")
print(f"    File size: {output_file_cognitive_first.stat().st_size:,} bytes")


✓ First visit files saved:
  Biomarker-based: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/ad/strict_first_visit.csv
    Total rows: 28
    File size: 5,465 bytes

  Cognitive-only: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/ad/cognitive_first_visit.csv
    Total rows: 108
    File size: 12,706 bytes


In [18]:
# Summary: All Visits vs First Visit Only
print("\n" + "=" * 70)
print("SUMMARY: ALL VISITS VS FIRST VISIT ONLY")
print("=" * 70)

print(f"\nSTRICT AD (cognitive + biomarkers):")
print(f"  All visits: {len(df_ad)} rows")
print(f"  First visit only: {len(df_ad_first_visit)} rows")
print(f"  Reduction: {len(df_ad) - len(df_ad_first_visit)} rows")

print(f"\nCOGNITIVE-ONLY AD:")
print(f"  All visits: {len(df_ad_cognitive)} rows")
print(f"  First visit only: {len(df_ad_cognitive_first_visit)} rows")
print(f"  Reduction: {len(df_ad_cognitive) - len(df_ad_cognitive_first_visit)} rows")

print(f"\nSTORED INDEX SETS (available for downstream use):")
print(f"  idx_strict_all:          {len(idx_strict_all)} indices")
print(f"  idx_cognitive_only_all:  {len(idx_cognitive_only_all)} indices")
print(f"  idx_strict_first:        {len(idx_strict_first)} indices")
print(f"  idx_cognitive_only_first:{len(idx_cognitive_only_first)} indices")

print(f"\nAll files saved to: {output_dir}")


SUMMARY: ALL VISITS VS FIRST VISIT ONLY

STRICT AD (cognitive + biomarkers):
  All visits: 33 rows
  First visit only: 28 rows
  Reduction: 5 rows

COGNITIVE-ONLY AD:
  All visits: 236 rows
  First visit only: 108 rows
  Reduction: 128 rows

STORED INDEX SETS (available for downstream use):
  idx_strict_all:          33 indices
  idx_cognitive_only_all:  202 indices
  idx_strict_first:        28 indices
  idx_cognitive_only_first:83 indices

All files saved to: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/ad


## Excel Export with Highlighted Rows

Export the full dataset to an Excel file with row highlighting:
- **Dark red** = Strict AD criteria (cognitive scores **+** biomarkers)
- **Light red** = Cognitive-only AD criteria (mmstot ≤ 24 AND cdrglobal ≥ 1, but does NOT meet biomarker criteria)

In [19]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font

display_cols_xl = ['file', 'Repseudonym', 'visdate', 'scan_date', 'scan_year', 'prmdiag',
                'mmstot', 'cdrglobal', 'ratio_Abeta42_40',
                'ratio_Abeta42_phosphotau181']

df_export = df[display_cols_xl].copy()
df_export['ad_type'] = ''
for idx in idx_strict_all:
    df_export.at[idx, 'ad_type'] = 'STRICT'
for idx in idx_cognitive_only_all:
    df_export.at[idx, 'ad_type'] = 'COGNITIVE_ONLY'

wb = Workbook()
ws = wb.active
ws.title = 'AD Visits'

# Fills
dark_red_fill = PatternFill(start_color='ff0606', end_color='ff0606', fill_type='solid')
light_red_fill = PatternFill(start_color='ff8080', end_color='ff8080', fill_type='solid')
white_font = Font(color='FFFFFF', bold=False)
header_fill = PatternFill(start_color='333333', end_color='333333', fill_type='solid')
header_font = Font(color='FFFFFF', bold=True)

export_cols = display_cols_xl + ['ad_type']
for col_idx, col_name in enumerate(export_cols, 1):
    cell = ws.cell(row=1, column=col_idx, value=col_name)
    cell.fill = header_fill
    cell.font = header_font

for excel_row, (df_idx, row_data) in enumerate(df_export.iterrows(), 2):
    is_strict = df_idx in idx_strict_all
    is_cognitive = df_idx in idx_cognitive_only_all
    for col_idx, col_name in enumerate(export_cols, 1):
        value = row_data[col_name]
        if pd.isna(value):
            value = ''
        cell = ws.cell(row=excel_row, column=col_idx, value=value)
        if is_strict:
            cell.fill = dark_red_fill
            cell.font = white_font
        elif is_cognitive:
            cell.fill = light_red_fill

# Auto-adjust column widths
for col_idx, col_name in enumerate(export_cols, 1):
    max_len = len(str(col_name))
    for row in ws.iter_rows(min_row=2, max_row=min(100, ws.max_row), min_col=col_idx, max_col=col_idx):
        for cell in row:
            if cell.value:
                max_len = max(max_len, len(str(cell.value)))
    ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = min(max_len + 2, 40)

wb.save(output_excel)

print(f'✓ AD visits highlighted Excel saved to: {output_excel}')
print(f'  Total rows: {len(df_export)}')
print(f'  Dark red (strict AD): {len(idx_strict_all)}')
print(f'  Light red (cognitive-only): {len(idx_cognitive_only_all)}')
print(f'  File size: {output_excel.stat().st_size:,} bytes')

✓ AD visits highlighted Excel saved to: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/ad/highlighted.xlsx
  Total rows: 4605
  Dark red (strict AD): 33
  Light red (cognitive-only): 202
  File size: 230,813 bytes
